# 06 — Evaluation: Reproducible Paper-Quality Runs

**역할**: 4개 모델(`vanilla` / `periodic` / `trajectory` / **`trajectory_sync` = ours**)을 동일 MuJoCo Ant-v5 rollout protocol로 비교 + frequency controllability 분석 + paper figures 생성.

**산출물**: `results/table{1,2}_*.md`, `results/eval_results.npz`, `figures/eval_figure{1..5}_*.png`

**예상 소요**: **~75분** (Colab T4 GPU)
- Table 1 in-dist: 4 models × 20 seeds × 1000 steps ≈ 25분
- Table 2 sweep: 3 phase models × 5 freqs × 10 seeds × 1000 steps ≈ 50분


---

**평가 모델 (4개, `pcdp.configs.EXPERIMENT_CONFIGS`에 등록됨)**
- `vanilla` — Vanilla DP
- `periodic` — Periodic Phase
- `trajectory` — Phase Trajectory (sync 없는 ablation)
- `trajectory_sync` — **Phase Trajectory + Sync (ours)** (`phase_trajectory_sync_lambda0.12.pt`)

**평가 항목**
- Table 1: 4개 모델 in-distribution gait quality (`survival`, `reward/step`, `forward velocity`, `measured freq`)
- Table 2: 3개 phase 모델 frequency command tracking (`|freq_err|`, `PLV`, `reward/step`) — vanilla 제외
- Figure 1: 4개 모델 reward/step 비교
- Figure 2: 3개 phase 모델 reward/step vs commanded frequency
- Figure 3: commanded vs measured gait frequency (**trajectory + ours만**; periodic은 mode collapse(~1.2 Hz)로 plot 영역 밖이라 제외, annotation에 평균 measured freq 표시)
- Figure 4: zone-aggregated |freq err| + PLV
- Figure 5: 대표 rollout phase 시계열
- Raw arrays: `eval_results.npz`

---

이 노트북은 **실행 orchestration만 담당**합니다. 핵심 로직은 `pcdp/evaluation.py`(4개 모델 자동 로드 + `run_in_distribution_evaluation` + `run_frequency_sweep_evaluation` + table markdown export), `pcdp/sampling.py`(DDIM action chunk sampling + MuJoCo Ant rollout), `pcdp/phase.py`(target phase trajectory 생성 + Hilbert-derived measured phase + PLV / frequency error metrics), `pcdp/experiment_plots.py`(`plot_paper_figures` — ours 강조 styling + Figure 3에서 periodic mode collapse annotation)에 있습니다.

In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')


In [ ]:
# Colab dependency setup
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    print("Not running in Google Colab; skipping dependency installation and using the current environment.")
else:
    project_root = Path.cwd()
    requirements_path = project_root / "requirements.txt"
    install_command = [sys.executable, "-m", "pip", "install"]

    if requirements_path.exists():
        # requirements.txt pins the runtime stack; -e . installs this repo from pyproject.toml.
        install_command.extend(["-r", str(requirements_path), "-e", str(project_root)])
    else:
        # Fallback to pyproject.toml dependencies if requirements.txt is unavailable.
        install_command.extend(["-e", str(project_root)])

    subprocess.check_call(install_command)

    import gymnasium as gym
    import mujoco
    import minari
    import torch

    gym.make("Ant-v5").close()

    print(f"gymnasium={gym.__version__}")
    print(f"mujoco={mujoco.__version__}")
    print(f"minari={minari.__version__}")
    print(f"torch={torch.__version__}")
    print("Ant-v5 environment smoke check passed.")


## 1. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 2. Imports + Config


In [ ]:
import gymnasium as gym
import torch

from pcdp.configs import set_global_seed
from pcdp.dataset import load_project_data
from pcdp.evaluation import (
    build_eval_results_payload,
    build_frequency_sweep_protocol,
    load_evaluation_state,
    print_frequency_sweep_summary,
    print_table1_summary,
    run_frequency_sweep_evaluation,
    write_frequency_tracking_table_markdown,
    write_table1_summary_markdown,
    run_in_distribution_evaluation,
    save_eval_results_npz,
)
from pcdp.experiment_plots import plot_paper_figures

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

## 3. 데이터 및 체크포인트 로드

In [ ]:
data = load_project_data(DATA_DIR)
set_global_seed(data['seed'], deterministic=True)


In [ ]:
state = load_evaluation_state(data, device=device, checkpoints_dir=CHECKPOINTS_DIR)


## 4. 평가 프로토콜 정의

In [ ]:
DT = 0.05
MAX_STEPS = 1000
N_SEEDS_INDIST = 20
N_SEEDS_SWEEP = 10

freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=3, ood_iqr_scale=1.5)

print(f"In-dist freqs: {freq_protocol.in_freqs.round(3).tolist()}")
print(f"OOD freqs:     {freq_protocol.ood_freqs.round(3).tolist()}")
print(f"Sweep freqs:   {freq_protocol.sweep_freqs.round(3).tolist()}")
print(f"Zones:         {freq_protocol.zone_labels.tolist()}")


## 5. Table 1 — In-distribution performance

In [ ]:
env = gym.make('Ant-v5')
table1_results = run_in_distribution_evaluation(
    state,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_INDIST,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_table1_summary(state, table1_results, freq_hz=float(data['freq_window_mean']))
write_table1_summary_markdown(
    state,
    table1_results,
    RESULTS_DIR / 'table1_indist_quality.md',
    freq_hz=float(data['freq_window_mean']),
    interval='ci95',
)


## 6. Table 2 — Frequency command tracking

In [ ]:
freq_results = run_frequency_sweep_evaluation(
    state,
    freq_protocol,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_SWEEP,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_frequency_sweep_summary(data, freq_protocol, freq_results)
write_frequency_tracking_table_markdown(
    freq_protocol,
    freq_results,
    RESULTS_DIR / 'table2_frequency_tracking.md',
    interval='ci95',
)

## 7. Figures — Contribution-focused visualizations

In [ ]:
plot_paper_figures(
    table1_results,
    freq_results,
    freq_protocol,
    data,
    FIGURES_DIR,
    n_seeds_sweep=N_SEEDS_SWEEP,
    state=state,
    interval='ci95',
)

## 8. 결과 저장 — `eval_results.npz`

In [ ]:
eval_payload = build_eval_results_payload(
    data,
    table1_results,
    freq_protocol,
    freq_results,
    n_seeds_indist=N_SEEDS_INDIST,
    n_seeds_sweep=N_SEEDS_SWEEP,
)
save_eval_results_npz(eval_payload, RESULTS_DIR / 'eval_results.npz')